In [ ]:
import os
import pandas as pd
import gensim
import gensim.corpora as corpora
from gensim.models import CoherenceModel
import matplotlib.pyplot as plt 

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module='gensim')


In [ ]:
# Load the preprocessed complaints data into a DataFrame

df = pd.read_csv('../data/complaints_processed_full.csv')
df.head()



In [ ]:
# Create a list of tokenized narratives

tokenized = df['narrative'].str.split().tolist()


In [ ]:
# Build GenSim dictionary and corpus for topic modeling
dictionary = corpora.Dictionary(tokenized)
corpus = [dictionary.doc2bow(text) for text in tokenized]

In [ ]:
# Evaluate coherence scores for different numbers of topics to select the optimal number of topics

candidate_topics = [5, 10, 15, 20, 25, 30]
coherence_scores = []

for k in candidate_topics:
    lda_model = gensim.models.LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=10
    )
    coherence_model = CoherenceModel(
        model=lda_model,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v'
    )
    coherence_scores.append(coherence_model.get_coherence())
    print(f"num_topics={k}, coherence={coherence_scores[-1]:.4f}")

In [ ]:
# Plot coherence scores to visualize the optimal number of topics

plt.figure(figsize=(10, 5))
plt.plot(candidate_topics, coherence_scores, marker='o')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score (c_v)')
plt.title('LDA Coherence Score by Number of Topics')
plt.xticks(candidate_topics)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Train the final LDA model using the optimal number of topics based on coherence scores

optimal_k = candidate_topics[coherence_scores.index(max(coherence_scores))]
print(f"Selected num_topics: {optimal_k}")

lda_final = gensim.models.LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=optimal_k,
    random_state=42,
    passes=10
)

In [ ]:
# Inspect topic-word distributions

for idx, topic in lda_final.print_topics(num_words=10):
    print(f"Topic {idx}: {topic}")

In [ ]:
# Extract document-topic vectors and assign dominant topic to each document

import numpy as np

def get_doc_topic_vector(bow, model, num_topics):
    topic_dist = dict(model.get_document_topics(bow, minimum_probability=0.0))
    return [topic_dist.get(i, 0.0) for i in range(num_topics)]

topic_vectors = np.array([
    get_doc_topic_vector(bow, lda_final, optimal_k)
    for bow in corpus
])

topic_df = pd.DataFrame(
    topic_vectors,
    columns=[f'topic_{i}' for i in range(optimal_k)]
)
topic_df.head()